# IMT 573 - Lab 5 - Data Analysis

### Instructions

Before beginning this assignment, please ensure you have access to a working instance of Jupyter Notebooks with Python 3.

1. First, replace the “YOUR NAME HERE” text in the next cell with your own full name. Any collaborators must also be listed in this cell.

2. Be sure to include well-documented (e.g. commented) code cells, figures, and clearly written text  explanations as necessary. Any figures should be clearly labeled and appropriately referenced within the text. Be sure that each visualization adds value to your written explanation; avoid redundancy – you do no need four different visualizations of the same pattern.

3. Collaboration on problem sets and labs is fun, useful, and encouraged. However, each student must turn in an individual write-up in their own words as well as code/work that is their own. Regardless of whether you work with others, what you turn in must be your own work; this includes code and interpretation of results. The names of all collaborators must be listed on each assignment. Do not copy-and-paste from other students’ responses or code - your code should never be on any other student's screen or machine.

4. All materials and resources that you use (with the exception of lecture slides) must be appropriately referenced within your assignment.

Name: Kathleen Ashbaker

Collaborators: Responsible/ Ethical AI Use Disclaimer: The markdown text and commentary were organized with the assistance of OpenAI’s ChatGPT-5 model. All code generated was thoroughly tested, and its outputs were critically evaluated by the author for scientific appropriateness and accuracy in statistical analysis. All interpretations, reflections, and perspectives on comic books are entirely the author’s own and reflect their unique viewpoints, which no large language model model can replicate.

In this module, we have focused on exploring data. Visualization is a great way to do this.

In [an article](https://fivethirtyeight.com/features/women-in-comic-books/) published on fivethirtyeight.com, the authors discuss gender representation in comic books. The data also contains a host of other information about comic book superheros and villans. We will use this dataset in this lab.

The data is split across the two major comic book publishers in the US - DC and Marvel. The urls for each are below. Use these to import the data for each of the publishers into separate dataframes and then combine the two into a larger dataframe.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
url_dc = "https://raw.githubusercontent.com/fivethirtyeight/data/master/comic-characters/dc-wikia-data.csv"
url_marvel = "https://raw.githubusercontent.com/fivethirtyeight/data/master/comic-characters/marvel-wikia-data.csv"

In [3]:
dc_data = pd.read_csv(url_dc)
marvel_data = pd.read_csv(url_marvel)

In [4]:
data_all = pd.concat([dc_data, marvel_data])

### Problem 1: Inspection

First, inspect the datasets to help you get a sense of what is contained in the data. You can find an overview of the data [here](https://github.com/fivethirtyeight/data/tree/master/comic-characters). What do you notice? Where may there be some issues with the data?

This (concatonated) dataset is sourced from two Wikis( Marvel and DC Comics), which back up the article, Comic Books Are Still Made By Men, For Men And About Men" 
By Walt Hickey, 2014. 

In [5]:


import pandas as pd
import numpy as np

def safe_concat(left: pd.DataFrame | None, right: pd.DataFrame | None) -> pd.DataFrame:
    if isinstance(left, pd.DataFrame) and isinstance(right, pd.DataFrame):
        return pd.concat([left, right], ignore_index=True, sort=False)
    if isinstance(left, pd.DataFrame):
        return left.copy()
    if isinstance(right, pd.DataFrame):
        return right.copy()
    raise ValueError("Provide at least one DataFrame: dc_data and/or marvel_data.")

# Use existing `data_all` if defined, otherwise build it.
if "data_all" not in globals():
    data_all = safe_concat(globals().get("dc_data"), globals().get("marvel_data"))

print("=== High-level shape ===")
shapes = {
    "dc_data": (globals().get("dc_data").shape if "dc_data" in globals() else None),
    "marvel_data": (globals().get("marvel_data").shape if "marvel_data" in globals() else None),
    "data_all": data_all.shape,
}
for k, v in shapes.items():
    print(f"{k:12s}: {v}")

print("\n=== Column overlap / differences ===")
if "dc_data" in globals() and "marvel_data" in globals():
    dc_cols = set(dc_data.columns)
    mv_cols = set(marvel_data.columns)
    print("Common columns         :", sorted(dc_cols & mv_cols))
    print("Only in dc_data        :", sorted(dc_cols - mv_cols))
    print("Only in marvel_data    :", sorted(mv_cols - dc_cols))
else:
    print("Only combined DataFrame available; skipping per-source comparison.")

print("\n=== dtypes, #unique, %missing by column (combined) ===")
summary = (
    pd.DataFrame({
        "dtype": data_all.dtypes.astype(str),
        "nunique": data_all.nunique(dropna=True),
        "missing": data_all.isna().sum(),
        "missing_pct": (data_all.isna().mean() * 100).round(2)
    })
    .sort_values(["missing_pct", "nunique"], ascending=[False, True])
)
print(summary)

print("\n=== Head / Sample ===")
print(data_all.head(3))
print("\nRandom sample:")
print(data_all.sample(min(3, len(data_all)), random_state=42))

print("\n=== Duplicate checks ===")
dup_all = data_all.duplicated().sum()
print(f"Exact duplicate rows (combined): {dup_all}")
# Heuristics: look for likely keys to test uniqueness
likely_keys = [c for c in data_all.columns if any(k in c.lower() for k in ["id", "slug", "key", "code", "name"])]
for col in likely_keys:
    uniq = data_all[col].nunique(dropna=True)
    miss = data_all[col].isna().sum()
    print(f"- Candidate key '{col}': unique={uniq:,} / rows={len(data_all):,} | missing={miss:,}")

print("\n=== Object columns that look numeric (may indicate dirty types) ===")
obj_cols = data_all.select_dtypes(include="object").columns
coercion_report = []
for col in obj_cols:
    coerced = pd.to_numeric(data_all[col].astype(str).str.replace(r"[,$% ]", "", regex=True),
                            errors="coerce")
    if coerced.notna().mean() > 0.8 and coerced.notna().sum() > 0:
        coercion_report.append((col, round(coerced.notna().mean()*100,2)))
if coercion_report:
    for col, pct in sorted(coercion_report, key=lambda x: -x[1])[:15]:
        print(f"- '{col}' → numeric coercible for ~{pct}% of rows (stored as object).")

print("\n=== Categorical hygiene (case/whitespace variants) ===")
def cat_issues(series: pd.Series) -> dict:
    # Identify if lower-cased stripped versions collapse many distinct categories
    s = series.dropna().astype(str)
    if s.empty: 
        return {}
    collapsed = s.str.strip().str.lower()
    if s.nunique() > collapsed.nunique() and s.nunique() <= 2000:
        return {"original_unique": s.nunique(), "collapsed_unique": collapsed.nunique()}
    return {}

cat_cols = [c for c in data_all.columns if data_all[c].dtype == "object" and data_all[c].nunique(dropna=True) <= 2000]
for c in cat_cols:
    issues = cat_issues(data_all[c])
    if issues:
        print(f"- '{c}' may have case/spacing inconsistencies: {issues}")

print("\n=== Common placeholder values (as strings) that should be treated as NA ===")
placeholders = {"", "na", "n/a", "none", "null", "unk", "unknown", "?", "tbd", "—", "-"}
flagged = {}
for c in obj_cols:
    vals = set(x.strip().lower() for x in data_all[c].dropna().astype(str).unique()[:5000])
    hits = sorted(v for v in vals if v in placeholders)
    if hits:
        flagged[c] = hits
if flagged:
    for c, hits in flagged.items():
        print(f"- '{c}' contains placeholder(s): {hits}")

print("\n=== Date-like columns parseability ===")
date_like = [c for c in data_all.columns if any(k in c.lower() for k in ["date", "time", "year", "released"])]
for c in date_like:
    try:
        parsed = pd.to_datetime(data_all[c], errors="coerce", utc=True, infer_datetime_format=True)
        print(f"- '{c}': parse success rate = {parsed.notna().mean()*100:.2f}%")
    except Exception as e:
        print(f"- '{c}': parse attempt failed ({e})")

print("\n=== Numeric columns: basic stats & outlier counts (IQR>1.5) ===")
num = data_all.select_dtypes(include=[np.number])
if not num.empty:
    desc = num.describe().T
    # Outlier count per column using IQR rule
    outlier_counts = {}
    for c in num.columns:
        q1, q3 = num[c].quantile(0.25), num[c].quantile(0.75)
        iqr = q3 - q1
        if pd.notna(iqr) and iqr > 0:
            lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            outlier_counts[c] = int(((num[c] < lo) | (num[c] > hi)).sum())
        else:
            outlier_counts[c] = np.nan
    desc["outliers_iqr15"] = pd.Series(outlier_counts)
    print(desc.sort_values("outliers_iqr15", ascending=False).head(20))
else:
    print("No numeric columns detected.")

print("\n=== Cross-source consistency spot-check (if both sources exist) ===")
if "dc_data" in globals() and "marvel_data" in globals():
    shared_cols = sorted(set(dc_data.columns) & set(marvel_data.columns))
    print("Shared columns:", shared_cols[:25], "..." if len(shared_cols) > 25 else "")
    # Compare dtype alignment for shared columns
    mismatched = []
    for c in shared_cols:
        d1, d2 = dc_data[c].dtype, marvel_data[c].dtype
        if d1 != d2:
            mismatched.append((c, str(d1), str(d2)))
    if mismatched:
        print("Columns with mismatched dtypes between sources:")
        for c, d1, d2 in mismatched[:30]:
            print(f"- {c}: dc={d1}, marvel={d2}")
    else:
        print("No dtype mismatches detected for shared columns.")
else:
    print("Single-source data; skipping cross-source comparison.")

print("\n=== Quick takeaways (auto-generated hints) ===")
issues = []
if dup_all > 0:
    issues.append(f"{dup_all} duplicate combined rows")
high_missing = summary[summary["missing_pct"] >= 30]
if not high_missing.empty:
    issues.append(f"High missingness in: {', '.join(high_missing.index.tolist()[:10])}")
if coercion_report:
    issues.append("Some numeric-looking fields stored as strings")
if flagged:
    issues.append("Placeholder strings that should be NA")
if issues:
    print("* Potential issues: " + "; ".join(issues))
else:
    print("* No obvious red flags from the automated checks above.")


=== High-level shape ===
dc_data     : (6896, 13)
marvel_data : (16376, 13)
data_all    : (23272, 14)

=== Column overlap / differences ===
Common columns         : ['ALIGN', 'ALIVE', 'APPEARANCES', 'EYE', 'FIRST APPEARANCE', 'GSM', 'HAIR', 'ID', 'SEX', 'name', 'page_id', 'urlslug']
Only in dc_data        : ['YEAR']
Only in marvel_data    : ['Year']

=== dtypes, #unique, %missing by column (combined) ===
                    dtype  nunique  missing  missing_pct
GSM                object        6    23118        99.34
YEAR              float64       79    16445        70.66
EYE                object       26    13395        57.56
Year              float64       75     7711        33.13
HAIR               object       28     6538        28.09
ID                 object        5     5783        24.85
ALIGN              object        4     3413        14.67
APPEARANCES       float64      442     1451         6.23
SEX                object        6      979         4.21
FIRST APPEARANCE   obj

/var/folders/1s/bsdc29gj1x16g8m2s5syblpm0000gn/T/ipykernel_79074/3192328854.py:108: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(data_all[c], errors="coerce", utc=True, infer_datetime_format=True)
/var/folders/1s/bsdc29gj1x16g8m2s5syblpm0000gn/T/ipykernel_79074/3192328854.py:108: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(data_all[c], errors="coerce", utc=True, infer_datetime_format=True)


Anomalies with the Data set : 

Missing Data Percentages of Noteworthiness: 


YEAR / Year	70.66% / 33.13%	Incomplete but potentially useful when considered in the right context. 

ALIGN	14.67% missing	Good coverage overall; this is key to my eventual question below 
APPEARANCES	6.23% missing	Excellent coverage for quantitative analysis.
SEX, ALIVE	Low missingness in these columns could support demographic or status-based sub-analyses.

### Problem 2: Formulate a question

Next, formulate one data science question of interest that can be answered with this dataset. Be sure to comment on why this question in interesting and what you could learn from finding an answer to it.

I selected the following question to ask for this analysis: 
"Is there a relationship between the alignment of characters (e.g., hero, villain, neutral)
 and the number of appearances or popularity metrics in the dataset?"
 I ask this because I personally enjoy narratives from  anti-hero or morally gray characters those whose moral codes aren’t as clear-cut on the surface. Think of Magneto from Marvel or Bruce Wayne from DC Comics. Both have endured profoundly traumatic experiences that shaped who they became. This complexity makes for compelling storytelling, as their narratives are rife  with raw, yet intricate and multi-faceted realism in comparison than the idealized narratives of their  “good” characters counterparts. To me, these stories raise thought-provoking questions about whether these characters possess redeemable qualities, and how much of both their good and bad traits are within ourselves. 

### Problem 3: Data analysis

Next, practice using your data science skills to answer you question. Follow the outlined steps in your data science process.

#### (a) Try the easy solution first

After filtering any anomolous values, try using descriptive statistics to see if you can get a general sense of the answer to your question.

In [6]:
import numpy as np
import pandas as pd

# ---- 0) Get combined df ----
if "data_all" not in globals():
    if "dc_data" in globals() and "marvel_data" in globals():
        data_all = pd.concat([dc_data, marvel_data], ignore_index=True, sort=False)
    else:
        raise RuntimeError("Please define `data_all` or provide `dc_data` and `marvel_data`.")
df = data_all.copy()

# ---- 1) Find alignment-like columns & preview values ----
cand_cols = [c for c in df.columns if any(k in c.lower() for k in
    ["alignment", "align", "allegiance", "moral", "side"])]
print("Alignment-like candidates:", cand_cols or "(none found)")

def peek_unique(s, k=12):
    vc = (s.astype(str).str.strip().str.lower().replace({"nan": np.nan})
          .dropna())
    return vc.value_counts().head(k)

for c in cand_cols:
    print(f"\nTop values in '{c}':")
    print(peek_unique(df[c]))

# ---- 2) Pick the best alignment column automatically ----
def looks_like_alignment(series: pd.Series) -> int:
    vals = series.astype(str).str.lower().str.strip().unique().tolist()
    score = 0
    tokens = ("good","bad","neutral","hero","villain","anti-hero","antihero")
    for t in tokens:
        if any(t in v for v in vals):
            score += 1
    return score

if not cand_cols:
    raise KeyError("No alignment-like column detected. Please inspect your schema.")

ALIGN_COL = max(cand_cols, key=lambda c: looks_like_alignment(df[c]))
print(f"\nChosen alignment column: {ALIGN_COL!r} (heuristic)")

# ---- 3) Normalize alignment -> hero|villain|neutral|other ----
hero_syn = {"good","hero","heroes","superhero","protagonist"}
vill_syn = {"bad","villain","villains","antagonist","evil"}
neut_syn = {"neutral","unknown","grey","gray","anti-hero","antihero","ambiguous","mixed","uncertain"}

def norm_align(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    # direct matches
    if s in hero_syn:    return "hero"
    if s in vill_syn:    return "villain"
    if s in neut_syn:    return "neutral"
    # substring heuristics
    if "hero" in s and "anti" not in s: return "hero"
    if "anti-hero" in s or "antihero" in s: return "neutral"
    if "villain" in s or "evil" in s: return "villain"
    if "good" in s:  return "hero"
    if "bad" in s:   return "villain"
    if "neutral" in s: return "neutral"
    return "other"

df["_alignment_clean"] = df[ALIGN_COL].map(norm_align)

print("\nCounts by cleaned alignment:")
print(df["_alignment_clean"].value_counts(dropna=True))

# ---- 4) Choose metric column and clean ----
metric_candidates = [c for c in df.columns if any(k in c.lower() for k in
    ["appearances","num_appearances","total_appearances","popularity","popularity_score","fans"])]
if not metric_candidates:
    raise KeyError("No metric column found (e.g., 'appearances' or 'popularity').")

pref = ["appearances","num_appearances","total_appearances","popularity","popularity_score","fans"]
METRIC_COL = next(c for p in pref for c in metric_candidates if p in c.lower())
print(f"\nMetric column: {METRIC_COL!r}")

df["_metric_raw"] = pd.to_numeric(df[METRIC_COL], errors="coerce")
df = df[df["_metric_raw"].notna() & (df["_metric_raw"] >= 0)]

# Keep only hero|villain|neutral for analysis; keep 'other' for info
valid_df = df[df["_alignment_clean"].isin(["hero","villain","neutral"])].copy()
valid_df["_metric_log1p"] = np.log1p(valid_df["_metric_raw"])

print("\nAnalysis group sizes (must have ≥2 groups with n≥2):")
print(valid_df["_alignment_clean"].value_counts())

# ---- 5) Descriptives ----
def boot_ci(a, agg=np.median, n=2000, seed=42):
    if len(a)==0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    boots = [agg(rng.choice(a, size=len(a), replace=True)) for _ in range(n)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return agg(a), lo, hi

stats_rows = []
for k, sub in valid_df.groupby("_alignment_clean"):
    med, lo, hi = boot_ci(sub["_metric_raw"].values, agg=np.median)
    stats_rows.append({"alignment": k, "n": len(sub), "median": med, "median_95%_lo": lo, "median_95%_hi": hi})
summary_df = pd.DataFrame(stats_rows).sort_values(["alignment"])
print("\nSummary (median with 95% bootstrap CI):")
print(summary_df)

# ---- 6) Inference if possible ----
try:
    from scipy import stats
    groups = [g["_metric_log1p"].values for _, g in valid_df.groupby("_alignment_clean")]
    labels = [lab for lab, _ in valid_df.groupby("_alignment_clean")]
    # keep groups with n>=2
    groups, labels = zip(*[(g,l) for g,l in zip(groups,labels) if len(g)>=2]) if groups else ([],[])
    if len(groups) >= 2:
        kw_h, kw_p = stats.kruskal(*groups)
        N = sum(len(g) for g in groups)
        k = len(groups)
        eps2 = (kw_h - (k-1)) / (N - k) if N>k else np.nan
        print(f"\nKruskal–Wallis across {labels}: H={kw_h:.3f}, p={kw_p:.5f}, epsilon^2≈{eps2:.3f}")
    else:
        print("\n[Info] Still not enough distinct alignment groups for a valid test. Rely on descriptives above.")
except Exception as e:
    print(f"\n[Info] Skipping inference ({e}).")


Alignment-like candidates: ['ALIGN']

Top values in 'ALIGN':
ALIGN
bad characters        9615
good characters       7468
neutral characters    2773
reformed criminals       3
Name: count, dtype: int64

Chosen alignment column: 'ALIGN' (heuristic)

Counts by cleaned alignment:
_alignment_clean
villain    9615
hero       7468
neutral    2773
other         3
Name: count, dtype: int64

Metric column: 'APPEARANCES'

Analysis group sizes (must have ≥2 groups with n≥2):
_alignment_clean
villain    9076
hero       7007
neutral    2644
Name: count, dtype: int64

Summary (median with 95% bootstrap CI):
  alignment     n  median  median_95%_lo  median_95%_hi
0      hero  7007     6.0            6.0            6.0
1   neutral  2644     3.0            3.0            4.0
2   villain  9076     3.0            3.0            3.0

Kruskal–Wallis across ('hero', 'neutral', 'villain'): H=964.346, p=0.00000, epsilon^2≈0.051


Add Description Here 

#### (b) Check distributions

Look at distribution(s) of your data to determine if there are any patterns that are evident with respect to your question of interest.

When examining the distribution of character alignments, several patterns become immediately evident. The variable ALIGN was successfully identified and cleaned, resulting in four groups: villain (n = 9615), hero (n = 7468), neutral (n = 2773), and a small “other” category (n = 3). This shows that the dataset is heavily skewed toward villain and hero alignments, which together make up the majority of the sample.

The chosen metric column of "APPEARANCES ", was analyzed across these alignment categories using a Kruskal–Wallis H test, which is non-parametric alternative to ANOVA given the non-normal, skewed nature of appearance count data

This statistically significant result indicates that there are meaningful differences in the number of appearances across alignment groups. In other words, whether a character is a hero, villain, or neutral is associated with differences in how often they appear in the comics.

Given the distribution counts and effect size (ε² = 0.051, a small-to-moderate effect), it’s clear that:

Villains are the most represented group numerically.

Heroes are also highly represented, though less so than villains.

Neutral characters form a noticeably smaller group, which may influence statistical power.

“Other” is too small to analyze meaningfully and can be excluded from further inferential testing.

These distributional patterns provide a strong foundation for further statistical and storytelling analyses—particularly around why villains may dominate the dataset in terms of representation and whether this reflects broader cultural or narrative trends in comics.

Narrative significance: Heroes may appear more frequently due to protagonist-centered storytelling, but anti-heroes or villains can drive equally strong engagement.

Fandom and popularity metrics: Character alignment could correlate with cultural prominence, e.g., how much villains like Joker or Magneto dominate narratives.

Potential insight: Finding systematic differences in appearances may reflect deeper storytelling trends in superhero media (e.g., moral binaries vs. complex character arcs).

There is a statistically significant difference in popularity (as measured by appearances) across alignment groups, with villains and heroes forming the dominant clusters in the dataset.


#### (c) What's next?

Considering what you've learned during this module, what could be additional steps to take to answer your question? Are there any potential issues with these steps? (In this case, it is perfectly acceptable to note issues without providing potential solutions).

To deepen the analysis of how character alignment relates to popularity, the next steps should extend beyond initial descriptive statistics and non-parametric group comparisons. First, conducting post-hoc pairwise comparisons (e.g., Dunn’s test)  would help identify which specific alignment groups (hero, villain, neutral) differ significantly from each other in their appearance counts. 

Second, integrating temporal and contextual factors such as year of first appearance, publisher (e.g., Marvel vs. DC), or major cultural events could  possibly reveal whether certain alignments dominate during specific decades, periods in time and or narrative trends. For example, villains might appear more frequently during certain decades, reflecting editorial strategies or broader socio-political narratives in media. Additionally, including demographic or attribute variables (e.g., gender, character status) could help uncover patterns of representation and visibility within these alignment groups.

Finally, developing visual analytic models, such as time series plots, alignment-based stratified histograms, or network centrality graphs (e.g., villain-hero interaction networks), would enable a more nuanced understanding of how alignment influences narrative presence over time and across storylines.

Caveats: 

Missing Data and Representation Bias: High missingness in some columns (e.g., GSM, EYE, YEAR) may limit multivariate modeling and could introduce bias if not handled carefully.

Right-Skewed Distributions: Appearance counts are heavily skewed, meaning a few characters dominate the data. This requires appropriate transformations or robust statistical approaches to avoid misleading inferences. Hence, we use non parametric statistical analyses for these situations. 

Ambiguity of Alignment Labels: Alignment categories may shift over time (e.g., characters evolving from villain to anti-hero), which static labels cannot fully capture.

Publisher Effects and External Influences: Patterns may reflect editorial and cultural priorities rather than inherent narrative properties, which complicates interpretation.

 Summary:
The next analytical phase involves pairwise testing, temporal and publisher-level stratification, and advanced visualization to clarify how alignment influences narrative prominence. Careful consideration of missingness, skewness, and conceptual ambiguity will be essential to ensure valid and interpretable findings.